<a href="https://colab.research.google.com/github/SasanduG/CV-Assignment-3/blob/main/CV_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 1

In [ ]:

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

batch_size = 50

trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, num_workers=0
)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=batch_size, shuffle=False, num_workers=0
)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Device: cuda


  4%|▎         | 6.16M/170M [02:39<1:19:21, 34.5kB/s]

In [ ]:

Din = 3 * 32 * 32
H = 100
K = 10

# Xavier initialization
W1 = torch.randn(Din, H, device=device) * (2 / (Din + H))**0.5
b1 = torch.zeros(H, device=device)

W2 = torch.randn(H, K, device=device) * (2 / (H + K))**0.5
b2 = torch.zeros(K, device=device)

lr = 0.1
lr_decay = 0.9
epochs = 10


In [ ]:

def forward(x):
    # Hidden layer
    z1 = x @ W1 + b1
    h = torch.sigmoid(z1)

    # Output layer
    logits = h @ W2 + b2

    return h, logits


Q3) b)

In [ ]:

def train_step(inputs, labels, lr):
    global W1, b1, W2, b2

    N = inputs.shape[0]
    x = inputs.reshape(N, -1)

    with torch.no_grad():
        # Forward propagation
        h, logits = forward(x)

        # Cross-entropy loss
        loss = F.cross_entropy(logits, labels)

        # Gradient of cross-entropy w.r.t. logits
        probs = torch.softmax(logits, dim=1)
        dlogits = probs.clone()
        dlogits[torch.arange(N, device=device), labels] -= 1
        dlogits /= N

        # Gradients for output layer
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(dim=0)

        # Gradients for hidden layer
        dh = dlogits @ W2.T
        dz1 = dh * h * (1 - h)

        dW1 = x.T @ dz1
        db1 = dz1.sum(dim=0)

        # Parameter updates
        W1 -= lr * dW1
        b1 -= lr * db1

        W2 -= lr * dW2
        b2 -= lr * db2

    return loss.item()


In [ ]:

def evaluate(loader):
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            x = inputs.reshape(inputs.shape[0], -1)

            h, logits = forward(x)

            loss = F.cross_entropy(logits, labels)
            total_loss += loss.item() * labels.size(0)

            predicted = torch.argmax(logits, dim=1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    average_loss = total_loss / total

    return average_loss, accuracy


In [ ]:

train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(epochs):

    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        train_step(inputs, labels, lr)

    # Evaluate using current parameters
    train_loss, train_acc = evaluate(trainloader)
    test_loss, test_acc = evaluate(testloader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )

    lr *= lr_decay


In [ ]:

epochs_range = range(1, epochs + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, test_losses, label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accuracies, label="Train Accuracy")
plt.plot(epochs_range, test_accuracies, label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"Final Training Accuracy: {train_accuracies[-1]:.2f}%")
print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")


## Question 2

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

transform_mnist = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

batch_size = 128

trainset_mnist = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform_mnist
)

testset_mnist = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform_mnist
)

trainloader_mnist = torch.utils.data.DataLoader(
    trainset_mnist,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

testloader_mnist = torch.utils.data.DataLoader(
    testset_mnist,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

print("Training images:", len(trainset_mnist))
print("Test images:", len(testset_mnist))


In [ ]:

class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5)

        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)
        self.activation = nn.Tanh()

        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.activation(self.conv1(x))
        x = self.pool(x)

        x = self.activation(self.conv2(x))
        x = self.pool(x)

        x = self.activation(self.conv3(x))

        x = torch.flatten(x, 1)

        x = self.activation(self.fc1(x))
        x = self.fc2(x)

        return x


model_lenet = LeNet5().to(device)

print(model_lenet)


Define the loss function and optimizer

In [ ]:

criterion = nn.CrossEntropyLoss()

learning_rate = 0.001
epochs = 10

optimizer_lenet = optim.Adam(
    model_lenet.parameters(),
    lr=learning_rate
)


Define the training function

In [ ]:

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward propagation
        outputs = model(images)

        # 3. Calculate loss
        loss = criterion(outputs, labels)

        # 4. Backpropagation
        loss.backward()

        # 5. Update parameters
        optimizer.step()


Define the evaluation function

In [ ]:

def evaluate_model(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            predictions = torch.argmax(outputs, dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = 100 * correct / total

    return avg_loss, accuracy


Train LeNet-5 for 10 epochs

In [ ]:

history_lenet = {
    "train_loss": [],
    "test_loss": [],
    "train_acc": [],
    "test_acc": []
}

for epoch in range(epochs):

    train_one_epoch(
        model_lenet,
        trainloader_mnist,
        optimizer_lenet,
        criterion
    )

    train_loss, train_acc = evaluate_model(
        model_lenet, trainloader_mnist, criterion
    )

    test_loss, test_acc = evaluate_model(
        model_lenet, testloader_mnist, criterion
    )

    history_lenet["train_loss"].append(train_loss)
    history_lenet["test_loss"].append(test_loss)
    history_lenet["train_acc"].append(train_acc)
    history_lenet["test_acc"].append(test_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Acc: {test_acc:.2f}%"
    )


Plot loss and accuracy

In [ ]:

epoch_numbers = range(1, epochs + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_numbers, history_lenet["train_loss"], label="Train")
plt.plot(epoch_numbers, history_lenet["test_loss"], label="Test")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("LeNet-5 Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epoch_numbers, history_lenet["train_acc"], label="Train")
plt.plot(epoch_numbers, history_lenet["test_acc"], label="Test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("LeNet-5 Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


Fnal accuracies

In [ ]:

print("LeNet-5 Results After 10 Epochs")
print("--------------------------------")
print(f"Training Accuracy: {history_lenet['train_acc'][-1]:.2f}%")
print(f"Test Accuracy:     {history_lenet['test_acc'][-1]:.2f}%")
